# Index Inclusion Probability Model

**Question:** at the moment a spinoff becomes effective, can we estimate the
probability that the child gets added to the S&P 500 **immediately**
(within 5 trading days), using only information available at that moment?

This matters for the long strategy in `spinoff_post_completion_strategy.ipynb`:
a child that is unlikely to be added keeps facing passive-fund selling
pressure after completion (a headwind for a long position), while a child
likely to be added avoids — or even reverses — that drag. `prob_included`
is a candidate signal for tilting/screening which children to hold long.

This notebook does **only** the modeling step: build the label, build the
features, fit the model, evaluate it honestly, and hand back a clean table
of per-event probabilities. Whether/how to fold `prob_included` into the
long strategy is a separate decision for a later notebook — this one is not
a backtest.

## Method: logistic regression, evaluated by leave-one-out cross-validation

**Why logistic regression.** The target is binary (included immediately /
not), and what we actually want out of the model is a *probability*, not
just a class label — logistic regression is the standard tool for that: it
fits a linear model in log-odds space,

$$\text{logit}(p) = \log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \dots + \beta_6 x_6,
\qquad p = P(\text{immediately included})$$

and its coefficients are directly interpretable (sign and, once features are
standardized, relative magnitude). With only a few features and a small
sample, a simple linear-in-log-odds model is the right level of complexity —
there isn't enough data to justify a more flexible, higher-variance model
(e.g. a tree ensemble).

**Why leave-one-out CV.** The full sample is small — see the count below —
so a conventional train/test split would waste data and give a noisy,
arbitrary estimate depending on which rows land in the test fold.
Leave-one-out (LOO) instead holds out **one event at a time**, refits the
model on all the others, and predicts that one held-out event. Repeating
this for every event gives an out-of-sample probability for *every* row
without ever letting a model see the event it's predicting — this is the
`prob_included` column in the final table, and it's what the metrics below
are computed from. It uses the data as efficiently as possible while still
being a genuine out-of-sample estimate.

**Why standardize features first.** Logistic regression coefficients are
only comparable to each other (to judge which feature matters most) if the
features are on the same scale — otherwise a feature measured in billions
would mechanically get a tiny coefficient next to one measured in single
digits. We standardize (zero mean, unit variance) via `StandardScaler`,
fit *inside* each LOO training fold so no information about the held-out
event's own scale leaks into its prediction.

## Features and lookahead audit

Every feature must be knowable at the effective date (when we'd actually
put a trade on) — nothing here is allowed to peek at what happens after.

| Feature | Definition | Known at effective date? |
|---|---|---|
| `log_child_mktcap` | log(child market cap at t=0 close) | Yes — t=0 close |
| `log_size_ratio` | log(child mktcap / that year's S&P 500 min-mktcap threshold) | Yes — t=0 close |
| `parent_index_weight` | parent's weight in the S&P 500 pre-spinoff | Yes — pre-effective |
| `forced_flow_adv` | passive AUM × parent weight ÷ parent ADV | Yes — pre-effective |
| `log_lag_days` | log(days from announcement to effective date) | Yes — known once announced |
| `passive_aum_B` | total passive S&P 500 AUM ($B) at announcement | Yes — pre-effective |

**Target:** `target = 1` if the child is added to the S&P 500 within 5
trading days of the effective date ("immediately included"), else `0`
(excluded, or added later — both are lumped together because both mean the
child does *not* get the immediate passive-inflow benefit).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

RAW_DIR   = Path('data/raw')
CLEAN_DIR = Path('data/clean')

## 1. Load data

In [2]:
events      = pd.read_csv(
    CLEAN_DIR / 'spinoff_events_merged.csv',
    parse_dates=['announce_date', 'effective_date', 'sp500_start', 'sp500_end']
)
children    = pd.read_parquet(RAW_DIR / 'spinoff_children_crsp.parquet')
child_sp500 = pd.read_parquet(RAW_DIR / 'spinoff_children_sp500.parquet')

print(f'Events loaded: {len(events)}')
print(f'Children CRSP: {children["permno"].nunique()} spinoff children, {len(children):,} rows')
print(f'S&P 500 membership records for children: {len(child_sp500)}')

Events loaded: 30
Children CRSP: 24 spinoff children, 17,945 rows
S&P 500 membership records for children: 14


## 2. Build the target label

A child is **immediately included** if its first S&P 500 addition date is
within 5 trading days of the spinoff's effective date. Everything else
(never added, or added later) is `0`.

In [3]:
first_inc = (
    child_sp500.sort_values('added_date')
    .groupby('child_ticker', as_index=False).first()
    [['child_ticker', 'added_date']]
)
eff_dates = events[['spinoff_ticker', 'effective_date']].rename(columns={'spinoff_ticker': 'child_ticker'})
first_inc = first_inc.merge(eff_dates, on='child_ticker', how='left')
first_inc['days_to_inclusion'] = (first_inc['added_date'] - first_inc['effective_date']).dt.days

def classify(row):
    if pd.isna(row['days_to_inclusion']):
        return 'excluded'
    return 'immediately included' if row['days_to_inclusion'] <= 5 else 'later included'

first_inc['inclusion_status'] = first_inc.apply(classify, axis=1)
status_map = first_inc.set_index('child_ticker')['inclusion_status'].to_dict()
events['inclusion_status'] = events['spinoff_ticker'].map(status_map).fillna('excluded')
events['target'] = (events['inclusion_status'] == 'immediately included').astype(int)

print(events['inclusion_status'].value_counts().to_string())
print(f'\nImmediate-inclusion rate: {events["target"].mean():.0%}')

inclusion_status
excluded                17
immediately included    13

Immediate-inclusion rate: 43%


## 3. Build features

`log_size_ratio` needs the S&P 500 minimum market-cap eligibility threshold
for the year the spinoff became effective — these are S&P Dow Jones
Indices' published minimums, hand-entered here since they aren't in a data
pull.

In [4]:
SP500_MIN_MKTCAP = {2020: 8.2e9, 2021: 11.8e9, 2022: 13.1e9, 2023: 12.7e9, 2024: 18.0e9}

def sp500_threshold(year):
    return SP500_MIN_MKTCAP.get(
        year, SP500_MIN_MKTCAP[min(SP500_MIN_MKTCAP, key=lambda y: abs(y - year))]
    )

child_t0 = (
    children.sort_values('date')
    .groupby('permno').first().reset_index()
    [['permno', 'mktcap', 'child_ticker']]
    .rename(columns={'mktcap': 'child_mktcap_t0'})
)

model_df = events.merge(
    child_t0[['child_ticker', 'child_mktcap_t0']],
    left_on='spinoff_ticker', right_on='child_ticker', how='left'
)

model_df['year'] = model_df['effective_date'].dt.year
model_df['size_threshold']  = model_df['year'].map(sp500_threshold)
model_df['log_child_mktcap'] = np.where(
    model_df['child_mktcap_t0'] > 0, np.log(model_df['child_mktcap_t0']), np.nan)
model_df['log_size_ratio'] = np.where(
    model_df['child_mktcap_t0'] > 0,
    np.log(model_df['child_mktcap_t0'] / model_df['size_threshold']), np.nan)
lag = (model_df['effective_date'] - model_df['announce_date']).dt.days
model_df['log_lag_days'] = np.where(lag > 0, np.log(lag), np.nan)
model_df['passive_aum_B'] = model_df['passive_aum_usd'] / 1e9

feature_cols = [
    'log_child_mktcap', 'log_size_ratio', 'parent_index_weight',
    'forced_flow_adv', 'log_lag_days', 'passive_aum_B',
]

model_clean = model_df.dropna(subset=feature_cols + ['target']).copy().reset_index(drop=True)
print(f'Usable sample: {len(model_clean)} events '
      f'(dropped {len(model_df) - len(model_clean)} for missing features)')
print(f'Target balance in usable sample: {dict(model_clean["target"].value_counts())} '
      f'(inclusion rate {model_clean["target"].mean():.0%})')

Usable sample: 21 events (dropped 9 for missing features)
Target balance in usable sample: {1: np.int64(13), 0: np.int64(8)} (inclusion rate 62%)


## 4. Fit with leave-one-out cross-validation

For each event: scale features using only the *other* events, fit logistic
regression on the other events, predict this event's probability. The
resulting `prob_included` is a genuine out-of-sample estimate for every row
— no event ever influences its own prediction.

In [5]:
X = model_clean[feature_cols].values
y = model_clean['target'].values

loo_probs = np.zeros(len(y))
loo_preds = np.zeros(len(y), dtype=int)

for train_idx, test_idx in LeaveOneOut().split(X):
    scaler = StandardScaler().fit(X[train_idx])
    X_train = scaler.transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])

    clf = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    clf.fit(X_train, y[train_idx])

    loo_probs[test_idx] = clf.predict_proba(X_test)[:, 1]
    loo_preds[test_idx] = clf.predict(X_test)

model_clean['prob_included'] = loo_probs
model_clean['pred_included'] = loo_preds
model_clean['correct'] = (model_clean['pred_included'] == model_clean['target'])

print('LOO-CV fitting complete.')

LOO-CV fitting complete.


## 5. Full-sample fit — for interpreting coefficients only

This second fit uses *all* events at once. It is **not** used to produce
`prob_included` above (that would leak information) — it exists only so we
can read off which features the model leans on.

In [6]:
scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(X)
clf_full = LogisticRegression(C=1.0, max_iter=500, random_state=42)
clf_full.fit(X_full, y)

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'standardized_coef': clf_full.coef_[0],
})
coef_df['abs_coef'] = coef_df['standardized_coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False).drop(columns='abs_coef')
coef_df['direction'] = np.where(coef_df['standardized_coef'] > 0,
                                 '+ → more likely included', '− → less likely included')

print('=== Standardized coefficients (full-sample fit, interpretation only) ===')
print(coef_df.to_string(index=False))

=== Standardized coefficients (full-sample fit, interpretation only) ===
            feature  standardized_coef                direction
   log_child_mktcap             0.9059 + → more likely included
     log_size_ratio             0.8671 + → more likely included
      passive_aum_B             0.3984 + → more likely included
parent_index_weight             0.3449 + → more likely included
    forced_flow_adv            -0.3386 − → less likely included
       log_lag_days             0.0954 + → more likely included


## 6. Metrics table

All metrics are computed from the LOO out-of-sample predictions
(`prob_included` / `pred_included` at a 0.5 threshold), not the full-sample
fit — this is the honest read of how well the model would have done
predicting each event without having seen its outcome.

In [7]:
majority_baseline = max(y.mean(), 1 - y.mean())
tn, fp, fn, tp = confusion_matrix(y, loo_preds).ravel()

metrics_table = pd.DataFrame([{
    'N (events)':            len(y),
    'Inclusion rate':        f'{y.mean():.0%}',
    'Majority-class baseline': f'{majority_baseline:.0%}',
    'LOO Accuracy':          f'{accuracy_score(y, loo_preds):.0%}',
    'Precision':             f'{precision_score(y, loo_preds, zero_division=0):.2f}',
    'Recall':                f'{recall_score(y, loo_preds, zero_division=0):.2f}',
    'F1':                    f'{f1_score(y, loo_preds, zero_division=0):.2f}',
    'AUC':                   f'{roc_auc_score(y, loo_probs):.3f}' if len(np.unique(y)) > 1 else 'n/a',
    'True Positives':        tp,
    'False Positives':       fp,
    'True Negatives':        tn,
    'False Negatives':       fn,
}]).T.rename(columns={0: 'value'})

print('=== LOO-CV Metrics ===')
display(metrics_table)

print('\nReading precision/recall here: "positive" = predicted immediately included.')
print(f'  Precision = {tp}/{tp+fp} = of the events the model called "included", '
      f'how many really were.')
print(f'  Recall    = {tp}/{tp+fn} = of the events truly included, '
      f'how many the model caught.')

=== LOO-CV Metrics ===


,value
N (events),21
Inclusion rate,62%
Majority-class baseline,62%
LOO Accuracy,71%
Precision,0.82
Recall,0.69
F1,0.75
AUC,0.750
True Positives,9
False Positives,2



Reading precision/recall here: "positive" = predicted immediately included.
  Precision = 9/11 = of the events the model called "included", how many really were.
  Recall    = 9/13 = of the events truly included, how many the model caught.


## 7. Full sample: probabilities and outcomes

One row per spinoff event: the features that went in, the LOO-CV
probability that came out, the resulting 0.5-threshold classification, the
true label, and whether the model got it right. Sorted by predicted
probability, highest first.

In [8]:
results_table = model_clean[[
    'spinoff_ticker', 'parent_ticker', 'effective_date',
    'log_child_mktcap', 'log_size_ratio', 'parent_index_weight',
    'forced_flow_adv', 'log_lag_days', 'passive_aum_B',
    'prob_included', 'pred_included', 'inclusion_status', 'target', 'correct',
]].sort_values('prob_included', ascending=False).reset_index(drop=True)

results_table_display = results_table.copy()
results_table_display['prob_included'] = (results_table_display['prob_included'] * 100).round(1)
results_table_display['pred_included']  = results_table_display['pred_included'].map({1: 'included', 0: 'not included'})
results_table_display = results_table_display.rename(columns={
    'spinoff_ticker': 'child', 'parent_ticker': 'parent',
    'prob_included': 'P(included) %', 'pred_included': 'model call',
    'inclusion_status': 'actual status', 'target': 'actual label', 'correct': 'model correct',
})

print(f'=== Full sample: {len(results_table_display)} events ===')
display(results_table_display)

=== Full sample: 21 events ===


,child,parent,effective_date,log_child_mktcap,log_size_ratio,parent_index_weight,forced_flow_adv,log_lag_days,passive_aum_B,P(included) %,model call,actual status,actual label,model correct
0,GEV,GE,2024-04-02,24.3706,0.7570,0.0041,11.4666,6.7742,"3,221.1740",98.2000,included,immediately included,1,True
1,VMW,DELL,2021-11-02,23.4288,0.2375,0.0001,16.6069,5.3083,"2,376.0377",96.5000,included,excluded,0,False
2,VTRS,PFE,2020-11-17,23.6992,0.8718,0.0063,7.1424,4.9488,"1,577.2549",95.2000,included,immediately included,1,True
3,GEHC,GE,2023-01-04,24.0376,0.7728,0.0027,13.0937,6.0426,"2,147.7157",92.8000,included,immediately included,1,True
4,OTIS,RTX,2020-04-03,23.7434,0.9160,0.0009,4.2174,6.2025,"1,260.2669",89.1000,included,immediately included,1,True
5,CEG,EXC,2022-02-02,23.5749,0.2790,0.0003,4.4019,5.8171,"2,368.5693",88.5000,included,immediately included,1,True
6,VLTO,DHR,2023-10-02,23.7653,0.5004,0.0048,20.1010,5.9480,"2,473.4455",85.4000,included,immediately included,1,True
7,SOLV,MMM,2024-04-01,23.2027,-0.4110,0.0013,7.0506,6.4216,"3,221.1740",84.2000,included,immediately included,1,True
8,CARR,RTX,2020-04-03,23.4081,0.5807,0.0009,4.2174,6.2025,"1,260.2669",81.2000,included,immediately included,1,True
9,KD,IBM,2021-11-04,22.5004,-0.6910,0.0027,7.7144,5.9713,"2,376.0377",81.1000,included,excluded,0,False


## Next step

This notebook stops at producing `prob_included` per event and evaluating
it honestly. Whether it's actually predictive of **post-completion returns**
(not just of the inclusion event itself) — and how to fold it into
`spinoff_post_completion_strategy.ipynb` as a screen or sizing tilt — is the
next thing to test, joining `results_table` onto the long-side trade ledger
by `child` / `spinoff_ticker`.